# Sprint 1 — Importación y exploración inicial de un dataset

**Dataset:** [House price prediction (Kaggle — `shree1992/housedata`)](https://www.kaggle.com/datasets/shree1992/housedata)

**Entorno:** Google Colab

---

## Índice
1. **Fase 1 — Importación de datasets** (desde el sistema local y desde Kaggle)
2. **Fase 2 — Importación desde Google Drive**
3. **Fase 3 — Análisis exploratorio inicial (EDA)**
4. **Fase 4 — Planteamiento de preguntas de investigación**

> **Nota sobre cómo ejecutar el cuaderno:** está pensado para **Google Colab**.
> Algunas celdas (subida de ficheros, API de Kaggle, montaje de Google Drive)
> usan utilidades propias de Colab (`google.colab`). Cada fase es independiente:
> con que uno de los métodos de carga funcione, el `DataFrame` queda disponible
> para el resto del cuaderno.


## Configuración inicial

Importamos las librerías que usaremos en todo el cuaderno. En Colab ya vienen
preinstaladas `pandas`, `numpy`, `matplotlib` y `seaborn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Opciones de visualización para ver bien las tablas
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

print("Librerías importadas correctamente")
print("Versión de pandas:", pd.__version__)


---
# Fase 1 — Importación de datasets

**Objetivo:** cargar el dataset desde **dos orígenes distintos**:

1. **Desde el sistema local** (subiendo el fichero `data.csv` desde nuestro ordenador).
2. **Desde Kaggle** (descargándolo directamente con la API de Kaggle).

El fichero original del dataset se llama **`data.csv`** y contiene información sobre
viviendas (precio, número de habitaciones, superficie, ubicación, etc.).


### 1.1 — Importación desde el sistema local

En Colab no tenemos acceso directo a las carpetas de nuestro ordenador, así que
usamos el módulo `google.colab.files` para **subir** el fichero manualmente. Al
ejecutar la celda aparecerá un botón **"Elegir archivos"**; seleccionamos el
`data.csv` que hemos descargado previamente desde Kaggle a nuestro equipo.


In [1]:
# --- Carga desde el sistema local (Google Colab) ---
from google.colab import files
import pandas as pd

# Abre el diálogo para seleccionar el fichero data.csv desde nuestro ordenador
uploaded = files.upload()

# files.upload() devuelve un diccionario {nombre_fichero: contenido_en_bytes}.
# Tomamos el nombre del primer (y único) fichero subido.
nombre_fichero = next(iter(uploaded))
print("Fichero subido:", nombre_fichero)

# Lo leemos con pandas
df_local = pd.read_csv(nombre_fichero)
print("Dimensiones (filas, columnas):", df_local.shape)
df_local.head()


Saving data.csv to data.csv
Fichero subido: data.csv


NameError: name 'pd' is not defined

**Comentario:** `files.upload()` carga el fichero en la memoria de la sesión de
Colab. Una vez subido, `pd.read_csv()` lo convierte en un `DataFrame`. Este método
es cómodo para ficheros pequeños y puntuales, pero el fichero **se pierde al
reiniciar el entorno**, por lo que habría que volver a subirlo.


### 1.2 — Importación desde Kaggle (API)

Descargar el dataset directamente desde Kaggle evita tener que subirlo a mano y
hace el cuaderno **reproducible**. Para usar la API de Kaggle necesitamos un token
de acceso:

1. Entramos en Kaggle → *Account* → sección **API** → **Create New API Token**.
2. Kaggle descarga un fichero **`kaggle.json`** con nuestras credenciales.
3. Lo subimos a Colab y lo colocamos en la ruta que espera la librería (`~/.kaggle/`).


In [ ]:
# --- Paso 1: subir el token kaggle.json ---
from google.colab import files
print("Sube tu fichero kaggle.json (Kaggle > Account > Create New API Token)")
files.upload()


In [ ]:
# --- Paso 2: colocar el token en la ruta esperada y dar permisos ---
import os

os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)  # permisos privados (requerido por Kaggle)
print("Token configurado correctamente")


In [ ]:
# --- Paso 3: descargar y descomprimir el dataset ---
# -d  : identificador del dataset   |   --unzip: descomprime automáticamente
!kaggle datasets download -d shree1992/housedata --unzip -p /content/housedata

# Vemos qué fichero(s) se han descargado
import os
print(os.listdir("/content/housedata"))


In [ ]:
# --- Paso 4: leer el CSV descargado desde Kaggle ---
df_kaggle = pd.read_csv("/content/housedata/data.csv")
print("Dimensiones (filas, columnas):", df_kaggle.shape)
df_kaggle.head()


**Comentario:** la API de Kaggle es el método más **reproducible**: cualquiera
con un token puede regenerar el dataset sin depender de un fichero local. El
inconveniente es la configuración inicial del token.

A partir de aquí trabajaremos con un único `DataFrame` llamado `df`. Usamos el
obtenido por Kaggle y, si no estuviera disponible, el de la carga local.


In [ ]:
# DataFrame de trabajo para el resto del cuaderno
try:
    df = df_kaggle.copy()
    print("Usando el dataset descargado de Kaggle")
except NameError:
    df = df_local.copy()
    print("Usando el dataset subido desde el sistema local")

print("Dimensiones del dataset de trabajo:", df.shape)


---
# Fase 2 — Importación desde Google Drive

**Objetivo:** investigar y documentar cómo importar el mismo fichero desde **Google
Drive**, una fuente muy habitual cuando trabajamos en Colab.

### Investigación del proceso
Tras consultar la documentación oficial de Colab, el procedimiento es:

1. **Montar** la unidad de Drive en el sistema de ficheros de Colab con
   `google.colab.drive.mount()`. Colab pedirá autorización para acceder a nuestra
   cuenta de Google.
2. Una vez montada, Drive aparece como una carpeta más en
   `/content/drive/MyDrive/...`, y podemos leer cualquier fichero con `pandas`
   usando su ruta.

**Requisito previo:** haber guardado `data.csv` en una carpeta de nuestro Drive
(por ejemplo `MyDrive/datasets/data.csv`).


In [ ]:
# --- Paso 1: montar Google Drive ---
from google.colab import drive

# Pide autorización y monta la unidad en /content/drive
drive.mount("/content/drive")


In [ ]:
# --- Paso 2: leer el fichero desde la ruta de Drive ---
# Ajusta la ruta a la carpeta donde hayas guardado data.csv en tu Drive.
ruta_drive = "/content/drive/MyDrive/datasets/data.csv"

df_drive = pd.read_csv(ruta_drive)
print("Dimensiones (filas, columnas):", df_drive.shape)
df_drive.head()


### Explicación y justificación

- **¿Por qué funciona?** Al montar Drive, Colab crea un punto de acceso
  (`/content/drive`) que enlaza nuestra unidad con la máquina virtual. Desde ese
  momento Drive se comporta como un disco local más, por lo que `pd.read_csv()`
  puede leer el fichero directamente con su ruta.

- **Ventajas frente a los métodos anteriores:**
  - El fichero **persiste** entre sesiones (no hay que volver a subirlo como en la
    carga local).
  - No requiere configurar tokens como la API de Kaggle.
  - Permite además **guardar** resultados (gráficos, CSV procesados) de vuelta en Drive.

- **Inconvenientes:** depende de nuestra cuenta de Google y de tener el fichero
  organizado en una ruta conocida; además, la primera vez requiere autorizar el acceso.

- **Capacidad de adaptación:** los tres métodos (local, Kaggle, Drive) producen el
  mismo `DataFrame`. Esto muestra que el origen de los datos es intercambiable y que
  el análisis posterior no depende de cómo se hayan cargado.


---
# Fase 3 — Análisis exploratorio inicial (EDA)

**Objetivo:** entender la estructura del dataset: qué variables contiene, de qué
tipo son, qué significan y si existen valores nulos o anómalos. Esto nos prepara
para los análisis de fases posteriores.


### 3.1 — Primer vistazo: dimensiones y muestras

In [ ]:
print("El dataset tiene {} filas y {} columnas\n".format(*df.shape))
df.head()


In [ ]:
df.tail()

### 3.2 — Tipos de variables e información general

In [ ]:
# Tipo de cada columna, nº de valores no nulos y memoria usada
df.info()


In [ ]:
# Clasificación rápida de columnas por tipo
numericas = df.select_dtypes(include=np.number).columns.tolist()
categoricas = df.select_dtypes(exclude=np.number).columns.tolist()

print("Variables numéricas ({}):".format(len(numericas)), numericas)
print("\nVariables categóricas / texto ({}):".format(len(categoricas)), categoricas)


### 3.3 — Diccionario de variables (significado)

| Variable | Tipo | Significado |
|---|---|---|
| `date` | fecha/texto | Fecha de registro de la venta |
| `price` | numérica | **Precio de la vivienda (variable objetivo)** |
| `bedrooms` | numérica | Número de dormitorios |
| `bathrooms` | numérica | Número de baños (admite decimales: aseos) |
| `sqft_living` | numérica | Superficie habitable (pies cuadrados) |
| `sqft_lot` | numérica | Superficie de la parcela (pies cuadrados) |
| `floors` | numérica | Número de plantas |
| `waterfront` | numérica (0/1) | Si la vivienda da al agua (1) o no (0) |
| `view` | numérica | Calidad de las vistas (escala 0–4) |
| `condition` | numérica | Estado de conservación (escala 1–5) |
| `sqft_above` | numérica | Superficie sobre el nivel del suelo |
| `sqft_basement` | numérica | Superficie del sótano |
| `yr_built` | numérica | Año de construcción |
| `yr_renovated` | numérica | Año de renovación (0 si nunca se renovó) |
| `street` | texto | Dirección de la vivienda |
| `city` | texto | Ciudad |
| `statezip` | texto | Estado y código postal |
| `country` | texto | País |

> Nota: las superficies vienen en **pies cuadrados** (1 m² ≈ 10,76 ft²).


### 3.4 — Estadísticos descriptivos

In [ ]:
# Resumen estadístico de las variables numéricas
df.describe().T


In [ ]:
# Resumen de las variables categóricas / texto
df.describe(include="object").T


### 3.5 — Análisis de valores nulos

In [ ]:
nulos = df.isnull().sum()
porcentaje = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({"nulos": nulos, "% nulos": porcentaje})
resumen_nulos = resumen_nulos[resumen_nulos["nulos"] > 0].sort_values("nulos", ascending=False)

if resumen_nulos.empty:
    print("No hay valores nulos explícitos (NaN) en el dataset.")
else:
    print("Columnas con valores nulos:")
    display(resumen_nulos)


In [ ]:
# Visualización de nulos por columna (mapa de calor)
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Mapa de valores nulos (amarillo = nulo)")
plt.tight_layout()
plt.show()


### 3.6 — Valores anómalos: más allá de los nulos

Aunque el dataset no tenga muchos `NaN`, conviene buscar **valores que no tienen
sentido**. En este dataset es conocido que algunas viviendas tienen `price = 0`
(precio no informado) y otras 0 dormitorios o 0 baños, lo que probablemente sean
errores de registro.

In [ ]:
print("Viviendas con precio = 0:", (df["price"] == 0).sum())
print("Viviendas con 0 dormitorios:", (df["bedrooms"] == 0).sum())
print("Viviendas con 0 baños:", (df["bathrooms"] == 0).sum())
print("Filas duplicadas:", df.duplicated().sum())


### 3.7 — Visualización de las variables principales

In [ ]:
# Distribución del precio (variable objetivo)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["price"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Distribución del precio")
axes[0].set_xlabel("Precio")

# El precio suele estar muy sesgado; lo vemos en escala logarítmica (precios > 0)
precios_validos = df.loc[df["price"] > 0, "price"]
sns.histplot(np.log10(precios_validos), bins=50, kde=True, ax=axes[1], color="orange")
axes[1].set_title("Distribución del precio (log10)")
axes[1].set_xlabel("log10(Precio)")
plt.tight_layout()
plt.show()


In [ ]:
# Histogramas de las principales variables numéricas
cols_interes = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "yr_built"]
cols_interes = [c for c in cols_interes if c in df.columns]

df[cols_interes].hist(bins=30, figsize=(14, 8), edgecolor="black")
plt.suptitle("Distribución de las variables numéricas principales")
plt.tight_layout()
plt.show()


In [ ]:
# Matriz de correlación entre variables numéricas
plt.figure(figsize=(11, 8))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Matriz de correlación (variables numéricas)")
plt.tight_layout()
plt.show()


In [ ]:
# ¿Qué variables se relacionan más con el precio?
correlacion_precio = (
    df.select_dtypes(include=np.number)
    .corr()["price"]
    .drop("price")
    .sort_values(ascending=False)
)
print("Correlación de cada variable con el precio:\n")
print(correlacion_precio)


In [ ]:
# Relación superficie habitable vs precio (una de las más correlacionadas)
plt.figure(figsize=(8, 5))
muestra = df[df["price"] > 0]
sns.scatterplot(data=muestra, x="sqft_living", y="price", alpha=0.3)
plt.title("Superficie habitable vs Precio")
plt.xlabel("sqft_living (pies cuadrados)")
plt.ylabel("Precio")
plt.tight_layout()
plt.show()


### 3.8 — Preparación del dataset para análisis posteriores

A partir del EDA, dejamos planteadas las acciones de limpieza (sin aplicarlas de
forma definitiva todavía, ya que serán objeto de fases siguientes):

- **Eliminar / imputar** las viviendas con `price = 0`, por ser registros no fiables.
- Revisar las viviendas con **0 dormitorios o 0 baños**.
- Convertir `date` a tipo **fecha** y, si interesa, extraer el año/mes de venta.
- Crear variables derivadas útiles, como la **antigüedad** de la vivienda
  (`año_actual - yr_built`) o si ha sido renovada (`yr_renovated > 0`).
- Separar de `statezip` el **código postal** para análisis por zona.

A continuación, una versión limpia básica de ejemplo (solo precios válidos):

In [ ]:
df_limpio = df[df["price"] > 0].copy()

# Ejemplos de variables derivadas
if "date" in df_limpio.columns:
    df_limpio["date"] = pd.to_datetime(df_limpio["date"], errors="coerce")
df_limpio["antiguedad"] = 2014 - df_limpio["yr_built"]  # el dataset es de 2014
df_limpio["renovada"] = (df_limpio["yr_renovated"] > 0).astype(int)

print("Filas tras eliminar precios = 0:", len(df_limpio), "(antes:", len(df), ")")
df_limpio[["price", "yr_built", "antiguedad", "yr_renovated", "renovada"]].head()


---
# Fase 4 — Planteamiento de preguntas de investigación

Tras la exploración inicial, planteamos preguntas que guiarán las fases siguientes.
Cada pregunta se vincula con variables concretas del dataset y con una hipótesis
contrastable.

### Preguntas e hipótesis

1. **¿Qué factores influyen más en el precio de una vivienda?**
   - *Variables:* `sqft_living`, `bathrooms`, `bedrooms`, `condition`, `view`, `waterfront`.
   - *Hipótesis:* la **superficie habitable** (`sqft_living`) es el factor con mayor
     correlación positiva con el precio, por encima del número de habitaciones.

2. **¿Tener vistas o estar frente al agua incrementa significativamente el precio?**
   - *Variables:* `waterfront`, `view`, `price`.
   - *Hipótesis:* las viviendas con `waterfront = 1` tienen un precio medio
     notablemente superior, aunque sean una minoría del dataset.

3. **¿Afecta la antigüedad de la vivienda a su precio? ¿Y haberla renovado?**
   - *Variables:* `yr_built`, `yr_renovated`, `price`.
   - *Hipótesis:* la antigüedad por sí sola tiene poco efecto, pero las viviendas
     **renovadas** se venden más caras a igualdad de antigüedad.

4. **¿Existen diferencias de precio entre ciudades?**
   - *Variables:* `city`, `statezip`, `price`.
   - *Hipótesis:* la **ubicación** (ciudad/código postal) explica una parte
     importante de la variación del precio.

5. **¿Es posible construir un modelo que prediga el precio a partir de las
   características de la vivienda?**
   - *Variables:* todas las numéricas relevantes como predictoras y `price` como objetivo.
   - *Hipótesis:* un modelo de regresión con las variables más correlacionadas
     ofrecerá una capacidad predictiva razonable, base para fases posteriores.

### Reflexión crítica
- El dataset tiene **limitaciones**: precios a 0, posibles errores en dormitorios/baños
  y datos concentrados en una zona geográfica concreta (área de Seattle, EE. UU.).
  Las conclusiones, por tanto, **no son generalizables** a otros mercados.
- Antes de modelar será imprescindible **limpiar y transformar** los datos
  (Fase 3.8) para evitar que los valores anómalos distorsionen los resultados.
- Las preguntas planteadas siguen un orden lógico: primero **entender** las
  relaciones (preguntas 1–4) y después **predecir** (pregunta 5).

---

## Conclusión del Sprint 1
Hemos importado el dataset desde **tres orígenes** (local, Kaggle y Google Drive),
realizado un **análisis exploratorio** de sus variables y valores nulos/anómalos, y
formulado **preguntas de investigación** con sus hipótesis. Con esto quedan sentadas
las bases para el análisis descriptivo y predictivo de las siguientes fases.
